# FER2013 Image Generator

## Emotion Face Classifier

To allow more seamless analysis between datasets, this notebook reads in the FER2013 csv data and then saves them as jpg images.

Images will be sorted by usage and emotion.

In [2]:
import os
import numpy as np
import pandas as pd
from PIL import Image

In [3]:
def check_directory_name(target_name) -> bool:
    """
    Check if the current directory name matches the target_name.
    If not, move up a directory and repeat the check.
    
    Args:
        target_name (str): The directory name to match.
        
    Returns:
        bool: True if the current directory name matches the target_name, False otherwise.
    """
    # Get the current directory path
    current_dir = os.getcwd()
    
    # Extract the directory name from the path
    current_dir_name = os.path.basename(current_dir)
    
    # Check if the current directory name matches the target_name
    if current_dir_name == target_name:
        print(f'Directory set to {current_dir}, matches target dir sting {target_name}.')
        return True
    else:
        # Move up a directory
        os.chdir('..')
        # Check if we have reached the root directory
        if os.getcwd() == current_dir:
            return False
        # Recursively call the function to check the parent directory
        return check_directory_name(target_name)

In [4]:
main_dir = 'EmotionFaceClassifier'
check_directory_name(main_dir)

Directory set to /Users/dsl/Documents/GitHub/EmotionFaceClassifier, matches target dir sting EmotionFaceClassifier.


True

In [5]:
from src.main import (
    convert_pixels_to_array,
    load_config
)

from src.fer2013_helpers import (
    create_img
)

In [6]:
df_path = 'data/fer2013/fer2013.csv'
df = pd.read_csv(df_path)

In [7]:
# Load common dicts from json config file
common_dicts = load_config('./configs/basics.json')
print(common_dicts.keys())

dict_keys(['usage_dict', 'emo_dict', 'frd_emo_dict', 'emo_color_dict', 'output_col_order', 'frd_output_col_order'])


In [8]:
# Load in key dicts from json for data mapping
emo_dict = common_dicts['emo_dict']

In [9]:
# Modify df for clarity
df = df.rename(columns={'emotion': 'emotion_id'})
df['emotion'] = df['emotion_id'].astype(str).map(emo_dict)

In [10]:
# Pixel data is read in as str, must be converted to np.array
df['image'] = df['pixels'].apply(convert_pixels_to_array)

In [11]:
df['usage']=df['Usage'].map(common_dicts['usage_dict'])
df['emo_count_id'] = df.groupby(['usage', 'emotion']).cumcount()+1

In [12]:
for _, row in df.iterrows():
    create_img(row)